# SmartSafe - Gıda Sektörü PPE Model Eğitimi

**Model:** YOLOv8m (medium) — Küçük datasette en iyi accuracy/hız dengesi
**Dataset:** PPE Food Manufacturing (476 görüntü, 5 sınıf)
**Sınıflar:** `Mask`, `Apron`, `gloves`, `Googles`, `Haircap`

### Adımlar:
1. Ortamı hazırla
2. Dataset'i yükle (Drive'dan)
3. Modeli eğit (yolov8m)
4. Sonuçları değerlendir
5. `best.pt` indir

In [ ]:
# ── 1. Ortam Kurulumu ──────────────────────────────────────────
!pip install ultralytics -q
!pip install roboflow -q

import ultralytics
ultralytics.checks()
print('✅ Ultralytics hazır')

In [ ]:
# ── 2. GPU Kontrolü ────────────────────────────────────────────
import torch
print(f'GPU mevcut: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️  GPU yok! Runtime > Change runtime type > T4 GPU seç')

In [ ]:
# ── 3. Dataset — SEÇENEK A: Google Drive'dan ──────────────────
# Önce bu hücreyi çalıştır, dataset'i Drive'a yüklediysen:
from google.colab import drive
drive.mount('/content/drive')

import os
# Dataset'in Drive'daki yolunu buraya yaz:
DATASET_PATH = '/content/drive/MyDrive/smartsafe_food_ppe_dataset'
print(f'Dataset yolu: {DATASET_PATH}')
print(f'Mevcut mu: {os.path.exists(DATASET_PATH)}')

In [ ]:
# ── 3. Dataset — SEÇENEK B: Direkt Roboflow'dan indir ─────────
# Drive yoksa veya dataset Drive'a yüklenmemişse bu hücreyi çalıştır:
from roboflow import Roboflow

rf = Roboflow(api_key='q2jdNWTJA6n61y2wccc9')
project = rf.workspace('rahma-5lrz6').project('ppe-food-manufacturing')
dataset = project.version(5).download('yolov8', location='/content/food_ppe_dataset')

DATASET_PATH = dataset.location
print(f'✅ Dataset indirildi: {DATASET_PATH}')

# data.yaml'ı göster
import yaml
with open(f'{DATASET_PATH}/data.yaml') as f:
    data_cfg = yaml.safe_load(f)
print(f'Sınıflar: {data_cfg.get("names", [])}')
print(f'Sınıf sayısı: {data_cfg.get("nc", "?")}')

In [ ]:
# ── 4. Model Eğitimi — YOLOv8m ────────────────────────────────
# yolov8m: 476 görüntü için optimal seçim
# n (nano) çok zayıf, l (large) overfitting riski var, m dengeli.

import yaml

# data.yaml içindeki mutlak yolları düzelt (Colab ortamında gerekli)
data_yaml_path = f'{DATASET_PATH}/data.yaml'
with open(data_yaml_path) as f:
    data_cfg = yaml.safe_load(f)

# Yolları Colab ortamına göre güncelle
data_cfg['train'] = f'{DATASET_PATH}/train/images'
data_cfg['val']   = f'{DATASET_PATH}/valid/images'
if 'test' in data_cfg:
    data_cfg['test'] = f'{DATASET_PATH}/test/images'

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_cfg, f)

print(f'data.yaml güncellendi: {data_cfg}')

In [ ]:
# ── 4b. Eğitimi Başlat ────────────────────────────────────────
!yolo train \
    data={DATASET_PATH}/data.yaml \
    model=yolov8m.pt \
    epochs=50 \
    imgsz=640 \
    batch=16 \
    patience=10 \
    augment=True \
    hsv_h=0.015 \
    hsv_s=0.7 \
    flipud=0.1 \
    name=smartsafe_food_ppe_v1 \
    project=/content/runs \
    device=0

# patience=10 → 10 epoch iyileşme olmazsa erken dur (overfitting önleme)
# augment=True → küçük dataset için veri artırma
# imgsz=640 → standart YOLO giriş boyutu

In [ ]:
# ── 5. Eğitim Sonuçlarını Değerlendir ─────────────────────────
import os
from ultralytics import YOLO

best_pt = '/content/runs/smartsafe_food_ppe_v1/weights/best.pt'
print(f'best.pt mevcut: {os.path.exists(best_pt)}')

if os.path.exists(best_pt):
    model = YOLO(best_pt)
    # Validation seti üzerinde değerlendir
    results = model.val(data=f'{DATASET_PATH}/data.yaml', device=0)
    print(f'\nmAP50: {results.box.map50:.3f}')
    print(f'mAP50-95: {results.box.map:.3f}')
    print(f'Precision: {results.box.mp:.3f}')
    print(f'Recall: {results.box.mr:.3f}')
    
    # Sınıf bazlı sonuçlar
    class_names = results.names
    print('\n── Sınıf Bazlı Sonuçlar ──')
    for i, name in class_names.items():
        print(f'  {name}: AP50={results.box.ap50[i]:.3f}')

In [ ]:
# ── 6. best.pt İndir ──────────────────────────────────────────
from google.colab import files
import shutil

best_pt = '/content/runs/smartsafe_food_ppe_v1/weights/best.pt'

# İndirme
files.download(best_pt)
print('✅ best.pt indirildi!')
print()
print('📋 Sonraki adım:')
print('   1. İndirdiğin best.pt dosyasını şuraya kopyala:')
print('   models/sh17_food_beverage/sh17_food_beverage_model/weights/best.pt')
print()
print('   2. .env dosyasında şu satırın dolu olduğunu kontrol et:')
print('   FOOD_PPE_LOCAL_MODEL=models/sh17_food_beverage/sh17_food_beverage_model/weights/best.pt')
print()
print('   3. Uygulamayı yeniden başlat — sistem otomatik yükler.')

In [ ]:
# ── 7. Drive'a Yedekle (opsiyonel) ────────────────────────────
import shutil
drive_dest = '/content/drive/MyDrive/SmartSafe_Models/food_ppe_v1'
os.makedirs(drive_dest, exist_ok=True)
shutil.copy(best_pt, f'{drive_dest}/best.pt')
print(f'✅ best.pt Drive\'a yedeklendi: {drive_dest}')